# Video Segmentation with Colour Histograms

This notebook investigates temporal video analysis using RGB colour histograms and histogram intersection.

Similarity between frames is used to examine visual changes over time and identify potential scene transitions.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Make sure outputs folder exists
os.makedirs("outputs", exist_ok=True)

# --- 1. Load video from Dataset B ---
cap = cv2.VideoCapture("datasetB.avi")
frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)

cap.release()
print("Number of frames loaded:", len(frames))

In [ ]:
# --- 2. Colour histogram function: returns (hist_R, hist_G, hist_B) ---
def colour_histogram(image, bins=256):
    # image is BGR from OpenCV
    b, g, r = cv2.split(image)

    hist_r = cv2.calcHist([r], [0], None, [bins], [0, 256]).flatten()
    hist_g = cv2.calcHist([g], [0], None, [bins], [0, 256]).flatten()
    hist_b = cv2.calcHist([b], [0], None, [bins], [0, 256]).flatten()

    return hist_r, hist_g, hist_b

In [ ]:
# --- 3. Compute histograms for every frame in the video ---
all_hists = [colour_histogram(f) for f in frames]

In [ ]:
idx1 = 0
idx2 = len(frames) // 2

frame1 = frames[idx1]
frame2 = frames[idx2]

h1_r, h1_g, h1_b = all_hists[idx1]
h2_r, h2_g, h2_b = all_hists[idx2]

In [ ]:
def save_frame(img, name):
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"outputs/{name}.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_hist(r, g, b, name):
    plt.plot(r, color="red")
    plt.plot(g, color="green")
    plt.plot(b, color="blue")
    plt.xlim([0,255])
    plt.xlabel("Intensity")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"outputs/{name}.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
save_frame(frame1, "3a_frame1")
save_frame(frame2, "3a_frame2")
save_hist(h1_r, h1_g, h1_b, "3a_hist1")
save_hist(h2_r, h2_g, h2_b, "3a_hist2")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)

video = cv2.VideoCapture("DatasetB.avi")

def load_frame(n):
    video.set(cv2.CAP_PROP_POS_FRAMES, n)
    ok, f = video.read()
    return f

# --- Histogram function ---
def colour_hist(img):
    hR = cv2.calcHist([img], [2], None, [256], [0,256])
    hG = cv2.calcHist([img], [1], None, [256], [0,256])
    hB = cv2.calcHist([img], [0], None, [256], [0,256])
    return hR, hG, hB

# --- Histogram intersection ---
def hist_intersection(h1, h2):
    return float(np.sum(np.minimum(h1, h2)))

# =============================
# 3(b) PART 1 — It and It+1
# =============================

t = 30
frame_t = load_frame(t)
frame_t1 = load_frame(t+1)

cv2.imwrite("outputs/3b_frame_t.png", frame_t)
cv2.imwrite("outputs/3b_frame_t1.png", frame_t1)

# Histograms
hR_t, hG_t, hB_t = colour_hist(frame_t)
hR_t1, hG_t1, hB_t1 = colour_hist(frame_t1)

# Plot histograms
def save_hist(hR, hG, hB, savepath, title):
    plt.figure(figsize=(6,4))
    plt.plot(hR, color="red", label="R")
    plt.plot(hG, color="green", label="G")
    plt.plot(hB, color="blue", label="B")
    plt.title(title)
    plt.xlabel("Intensity")
    plt.ylabel("Count")
    plt.xlim([0,255])
    plt.legend()
    plt.tight_layout()
    plt.savefig(savepath, dpi=300)
    plt.close()

save_hist(hR_t, hG_t, hB_t, "outputs/3b_hist_t.png", "Histogram of It")
save_hist(hR_t1, hG_t1, hB_t1, "outputs/3b_hist_t1.png", "Histogram of It+1")

# Intersection values (R,G,B separately)
inter_R = hist_intersection(hR_t, hR_t1)
inter_G = hist_intersection(hG_t, hG_t1)
inter_B = hist_intersection(hB_t, hB_t1)

with open("outputs/3b_intersection_values.txt", "w") as f:
    f.write(f"Intersection (R): {inter_R}\n")
    f.write(f"Intersection (G): {inter_G}\n")
    f.write(f"Intersection (B): {inter_B}\n")

# =============================
# 3(b) PART 2 — Intersection across entire video
# =============================

frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
intersections = []

for i in range(frame_count - 1):
    f1 = load_frame(i)
    f2 = load_frame(i+1)

    if f1 is None or f2 is None:
        break

    hR1, hG1, hB1 = colour_hist(f1)
    hR2, hG2, hB2 = colour_hist(f2)

    total_intersection = (
        hist_intersection(hR1, hR2) +
        hist_intersection(hG1, hG2) +
        hist_intersection(hB1, hB2)
    )
    intersections.append(total_intersection)
# Normalisation
intersections = np.array(intersections)
norm_intersections = (intersections - intersections.min()) / (intersections.max() - intersections.min())

# --- Raw intersection (bar plot) ---
plt.figure(figsize=(12,4))
plt.bar(range(len(intersections)), intersections, width=1.0)
plt.title("Histogram Intersection Over Time (Bar Plot)")
plt.xlabel("Frame Index")
plt.ylabel("Intersection Value")
plt.tight_layout()
plt.savefig("outputs/3b_intersection_barplot.png", dpi=300)
plt.close()

# --- Normalised intersection (bar plot) ---
plt.figure(figsize=(12,4))
plt.bar(range(len(norm_intersections)), norm_intersections, width=1.0)
plt.title("Normalized Histogram Intersection Over Time (Bar Plot)")
plt.xlabel("Frame Index")
plt.ylabel("Normalized Value (0–1)")
plt.tight_layout()
plt.savefig("outputs/3b_intersection_normalized_barplot.png", dpi=300)
plt.close()